# Lecture 1 — Practical: *Meeting Your Transcriptomic Dataset*
### Practical Machine Learning for Transcriptomics in Cancer Research

**No modelling today.** The whole point of this session is to *know your data before you trust it*.
You will load a real transcriptomic dataset, audit its clinical metadata, define a clean label,
measure class balance, run PCA, discover a batch effect, and build a defensible
train / validation / test split.

Adopt the mindset of an analyst writing the **"data sanity"** section of a methods paper — not
someone racing to a result. Every decision you make should be **written down** as you go.

---

#### The data (real cohorts)

| Cohort | Platform | Role | Source |
|---|---|---|---|
| **METABRIC** (~1,900 tumours) | Illumina HT-12 | primary cohort, long-term survival follow-up | cBioPortal `brca_metabric` |
| **GSE6532** (Loi et al.) | Affymetrix | second platform — combined in **deliberately** | NCBI GEO |

We predict **recurrence** (a relapse / DMFS-style **binary** endpoint). METABRIC has **no pCR** —
that's expected, and is itself a teaching point about letting the data choose the endpoint.

> **Why combine two cohorts on two platforms?** Because that is exactly where a real **batch
> effect** is born — and Section 4 is about discovering it. This is a *constructed teaching
> example*, not how you would assemble a genuine validation set.

> **Network note.** These cells download real data from cBioPortal and GEO. You need internet
> access. Downloads are cached to the lesson's `practical/task/datasets/` folder (resolved
> automatically), so you only fetch once. The data files are **git-ignored** — they are not
> committed to the repository; re-running this notebook re-fetches them on demand.


## Section 0 — Setup & the data loaders  *(read these)*

Environment and downloads are handled once by **Lesson 0** (`lessons/lesson00_prerequisites/`). The cell below imports the scientific stack and **shows you the loader functions** — how the course
**reads and parses** METABRIC (cBioPortal) and GSE6532 (NCBI GEO) from the cache. The *download* itself
is done once by Lesson 0; these loaders just read what's already there. Later lessons don't repeat this —
they load the cohort we checkpoint at the end.

In [1]:
# ── Setup — imports, the data loaders (revealed), and checkpoint save ──────────
# The environment (conda `ml26`) and the data download are handled ONCE by LESSON 0
# (lessons/lesson00_prerequisites/). This notebook installs and downloads NOTHING —
# the loaders below simply READ the files Lesson 0 already cached; if one is missing
# they point you back to Lesson 0. Read them: this is how the two cohorts are parsed.
import os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
RANDOM_STATE = 0
np.random.seed(RANDOM_STATE)

def _resolve_data_dir():
    """The shared data cache (…/lesson01_*/practical/task/datasets), populated by Lesson 0."""
    here = Path.cwd()
    for parent in [here, *here.parents]:
        for cand in sorted((parent / "lessons").glob("lesson01_*/practical/task/datasets")):
            return cand
    return here / "datasets"
DATA_DIR = str(_resolve_data_dir())

def _cached(fname):
    """Path to a file Lesson 0 downloaded; raises a clear pointer to Lesson 0 if it is missing."""
    p = os.path.join(DATA_DIR, fname)
    if not os.path.exists(p) or os.path.getsize(p) == 0:
        raise FileNotFoundError(
            f"'{fname}' is not in the data cache ({DATA_DIR}).\n"
            f"Run Lesson 0 first: lessons/lesson00_prerequisites/lesson00_prerequisites.ipynb")
    return p

def load_metabric():
    """METABRIC (Illumina microarray): read the cached cBioPortal files -> expr (genes x samples) + clinical."""
    expr = pd.read_csv(_cached("data_mrna_illumina_microarray.txt"), sep="\t", low_memory=False)
    expr = expr.drop(columns=[c for c in ["Entrez_Gene_Id"] if c in expr.columns])
    expr = expr.dropna(subset=["Hugo_Symbol"]).set_index("Hugo_Symbol")
    expr = expr[~expr.index.duplicated(keep="first")]
    pat = pd.read_csv(_cached("data_clinical_patient.txt"), sep="\t", comment="#", low_memory=False)
    smp = pd.read_csv(_cached("data_clinical_sample.txt"),  sep="\t", comment="#", low_memory=False)
    clin = pat.merge(smp, on="PATIENT_ID", how="inner", suffixes=("", "_smp"))
    if "SAMPLE_ID" in clin.columns:
        clin = clin.set_index("SAMPLE_ID")
    return expr, clin

def load_gse6532():
    """GSE6532 (Loi et al., Affymetrix): read the cached GEO SOFT file -> symbol-collapsed expr + clinical."""
    import GEOparse, collections
    gse = GEOparse.get_GEO(filepath=_cached("GSE6532_family.soft.gz"), silent=True)
    plat_of = {n: g.metadata.get("platform_id", ["?"])[0] for n, g in gse.gsms.items()}
    dom = collections.Counter(plat_of.values()).most_common(1)[0][0]     # dominant platform
    gpl = gse.gpls[dom].table
    sym = next((c for c in gpl.columns if c.lower() in ("gene symbol", "gene_symbol", "symbol")), None)
    pmap = gpl.set_index("ID")[sym].dropna().astype(str); pmap = pmap[pmap.str.len() > 0]
    cols, meta = {}, {}
    for n, g in gse.gsms.items():
        if plat_of[n] != dom:
            continue
        t = g.table
        if t is None or "VALUE" not in t.columns:
            continue
        cols[n] = pd.Series(t["VALUE"].values, index=t["ID_REF"].astype(str).values)
        row = {"title": "; ".join(g.metadata.get("title", []))}
        for it in g.metadata.get("characteristics_ch1", []):
            if ":" in it:
                k, v = it.split(":", 1); row[k.strip().lower()] = v.strip()
        meta[n] = row
    expr = pd.DataFrame(cols); expr = expr[expr.index.isin(pmap.index)]
    expr.index = pmap.loc[expr.index].values; expr = expr.groupby(level=0).mean()
    return expr, pd.DataFrame(meta).T

# ── checkpoint save — Lessons 2-5 LOAD what this notebook saves at the end ──────
import pickle
def save_checkpoint(name, **objs):
    d = Path(DATA_DIR) / "derived"; d.mkdir(parents=True, exist_ok=True)
    path = d / f"{name}.pkl"
    with open(path, "wb") as fh:
        pickle.dump(objs, fh)
    print(f"saved checkpoint '{name}'  ->  {path}")
    return path

print("Loaders ready (read-only): load_metabric(), load_gse6532()  |  DATA_DIR =", os.path.abspath(DATA_DIR))


Loaders ready (read-only): load_metabric(), load_gse6532()  |  DATA_DIR = /Users/valer/Desktop/Github/ml26/lessons/lesson01_biological_question/practical/task/datasets


### The prediction task (write it in your own words)

Before touching any code, state the prediction task **precisely**. A well-posed task names four
things: the **input**, the **output (label)**, **when** each is measured, and **on whom**.

> **Exercise 0.1 — edit the markdown below and commit your task statement.**


**My task statement:**

*(TODO — replace this with your own. Name the input, the output/label, when each is measured,
and on whom. Two or three sentences is plenty.)*

- **Input (X):** …
- **Output (y):** …
- **When:** …
- **On whom:** …


---
## Loading the data

The cell below fetches both cohorts using the shared loaders imported in Section 0 (the data itself is downloaded once by **Lesson 0**). Each returns expression as **genes × samples** plus aligned clinical metadata — you don't need to edit anything here; the teaching happens *after* the data is loaded.

In [2]:
# Fetch both cohorts (cached after first run).
metabric_expr, metabric_clin = load_metabric()
print("METABRIC expression (genes x samples):", metabric_expr.shape)
print("METABRIC clinical:", metabric_clin.shape)

gse_expr, gse_clin = load_gse6532()
print("GSE6532 expression (probes x samples):", gse_expr.shape)
print("GSE6532 clinical:", gse_clin.shape)

METABRIC expression (genes x samples): (20384, 1980)
METABRIC clinical: (2509, 35)
GSE6532 expression (probes x samples): (13515, 327)
GSE6532 clinical: (327, 18)


---
## Section 1 — Loading & orienting the data  *(≈20 min)*

The loaders gave you expression as **genes × samples** (the bioinformatics convention). The ML
convention is **samples × genes** (scikit-learn expects rows = examples). Transposition errors are
a classic bug — so you will determine the orientation *from evidence*, then assert it.


> **Exercise 1.1 — determine orientation from evidence, then assert it.**
>
> Don't just trust the docstring. Look at the shapes and the index/columns and *reason* about which
> axis is samples and which is genes. Then transpose METABRIC to **samples × genes** and assert it.
>
> *Hints:*
> - There are ~20,000 genes but only ~1,900 samples — the long axis is genes.
> - Gene names look like `ESR1`, `MKI67`; sample IDs look like `MB-0001`.
> - After transposing, `X.shape[0]` should be the number of samples (the smaller number).


In [ ]:
# TODO 1.1
# (a) Inspect metabric_expr: what do the index and columns look like? what is the shape?
print("rows (index) sample:", list(metabric_expr.index[:3]), "...")
print("cols sample:", list(metabric_expr.columns[:3]), "...")
print("shape:", metabric_expr.shape)

# (b) Decide which axis is samples and which is genes, and transpose to samples x genes.
#     X_metabric = ...

# (c) ASSERT your orientation (e.g. genes should outnumber samples; 'ESR1' among columns).
#     Also report value range and number of missing values.
# X_metabric = ...
# assert ...


> **Exercise 1.2 — join expression to clinical metadata, and investigate failures.**
>
> Join `X_metabric` (indexed by sample ID) to `metabric_clin` (also indexed by `SAMPLE_ID`).
> Report how many samples matched, and **look at** the ones that didn't — don't silently drop them.
>
> *Hint:* `X_metabric.index` vs `metabric_clin.index`; use set operations to see the mismatch.


In [ ]:
# TODO 1.2
# Compare the sample IDs in X_metabric.index vs metabric_clin.index.
# Report: how many matched? how many are expression-only / clinical-only?
# Then keep only matched samples, aligned in the same order, as X_metabric and clin.
#
# expr_ids = set(X_metabric.index); clin_ids = set(metabric_clin.index)
# ...


> **Discussion 1 (write a sentence).** What real-world events cause samples to fail a join
> (relabelled IDs, withdrawn consent, QC failures, version drift)? Why is *silently* dropping
> unmatched samples dangerous for the conclusions you'll later draw?


---
## Section 2 — Auditing the clinical metadata and the label  *(≈25 min)*

METABRIC has **no pCR**. It has long-term survival follow-up, so our label is **binary recurrence**.
You must turn messy survival fields into a clean target — and **record every decision**.


First, filter to the course cohort — **HR+/HER2−** — to match the clinical question.

> **Exercise 2.0 — filter to HR+/HER2−** using the receptor-status columns, and report how many
> tumours remain.
>
> *Hint:* METABRIC clinical columns include `ER_STATUS`, `PR_STATUS`, `HER2_STATUS`
> (values like `Positive`/`Negative`). HR+ = ER **or** PR positive; HER2− = HER2 negative.


In [ ]:
# TODO 2.0
# Look at ER_STATUS / PR_STATUS / HER2_STATUS, then build a boolean mask for HR+/HER2-.
# HR+ = ER positive OR PR positive ;  HER2- = HER2 negative.
# Apply the mask to BOTH clin and X_metabric, and report how many remain.
#
# for c in ["ER_STATUS", "PR_STATUS", "HER2_STATUS"]:
#     print(c, clin[c].value_counts(dropna=False))
# mask = ...


> **Exercise 2.1 — define a binary recurrence label, with an explicit censoring policy.**
>
> METABRIC exposes relapse-free survival fields (commonly `RFS_STATUS` like
> `"0:Not Recurred"` / `"1:Recurred"`, and `RFS_MONTHS`). Define:
>
> `y = 1` if the patient **recurred within the horizon** (e.g. 60 months);
> `y = 0` if the patient was **followed at least the horizon with no recurrence**;
> **exclude** patients censored *before* the horizon (we cannot know their outcome).
>
> Report how many are positive, negative, and excluded — and **write down** your horizon and
> censoring rule. *A censored patient is **not** automatically a non-event.*


In [ ]:
# TODO 2.1
# Pick a HORIZON (months). Find the relapse status & months columns (RFS_* or DFS_*).
# Build y: 1 = recurred within horizon; 0 = event-free through horizon;
#          NaN = censored before horizon (EXCLUDE -- unknown outcome).
# Apply the keep-mask to clin, X_metabric, y. Report positives / negatives / excluded.
#
# HORIZON = ...
# status_col = ...; months_col = ...
# recurred = ...; months = ...
# y = pd.Series(index=clin.index, dtype="float")
# ...


> **Exercise 2.2 — name a leaky field.** Identify at least one metadata column that must **not**
> be used as a feature because it would leak — anything measured *at or after* the outcome.
>
> *Hint:* survival/vital-status columns, the relapse fields themselves, or treatment given *in
> response to* progression. Write one sentence explaining why.


In [ ]:
# TODO 2.2
# List metadata columns that would leak the outcome if used as features, and say why in a comment.
# leaky = [c for c in clin.columns if ...]
# print(leaky)


> **Discussion 2 (write a few sentences).** The published model uses RFS for development and DMFS
> for external validation, arguing DMFS is the more faithful long-term endpoint. Why does the choice
> (any relapse vs distant-only) change *both* the event count and the clinical meaning? And: pCR
> was simply **unavailable** here — why is *"use the endpoint your data supports"* a defensible
> scientific decision rather than a compromise to hide?


---
## Section 3 — Class balance  *(≈15 min)*

Recurrence events are a **minority**. That has consequences for metrics (accuracy will mislead)
and for splitting (you must stratify).


> **Exercise 3.1 — quantify and visualise class balance**, and state the **smallest class count**
> (the number of recurrence events). Plot the two counts as a bar chart (Figure matches the deck's
> class-balance figure).
>
> **Exercise 3.2 (writing).** Before you test it in Section 5: predict, in a sentence, how a naive
> 80/20 random split could distribute the minority class badly. What is the worst case?


In [ ]:
# TODO 3.1 / 3.2
# Compute counts of y==0 and y==1, the proportions, and the smallest class count.
# Make a 2-bar bar chart. Then in a comment, note the accuracy of an 'always no-relapse' model.
# Finally answer Exercise 3.2 in the markdown cell below.
#
# counts = y.value_counts().sort_index()
# ...


**My Section 3.2 prediction (worst case of a naive random split):**

*(TODO in the student notebook — write your prediction here before Section 5.)*


---
## Section 4 — PCA and batch effects  *(≈40 min — a core section)*

Now we **combine METABRIC with GSE6532** on their shared genes. The two platforms (Illumina vs
Affymetrix) create a strong technical signal. PCA will let us *see* it.

First, build the combined matrix and a `batch` label. (This cell is shared infrastructure — read it,
then the exercises follow.)


In [ ]:
# GSE6532 is now symbol-indexed (genes x samples); orient to samples x genes and
# intersect with METABRIC on shared gene symbols -> the common feature space for PCA.
gse_X = gse_expr.T.copy()                       # samples x genes
common_genes = [g for g in X_metabric.columns if g in gse_X.columns]
print("common features (shared gene symbols) used for PCA:", len(common_genes))
assert len(common_genes) > 1000, "Expected thousands of shared genes after probe->symbol mapping."

# Build combined matrix on the shared genes. We deliberately DO NOT batch-correct here,
# so the cross-platform batch effect stays visible in the PCA.
A = X_metabric[common_genes].copy()
B = gse_X[common_genes].copy()
combined = pd.concat([A, B], axis=0)
batch = pd.Series(["METABRIC/Illumina"]*len(A) + ["GSE6532/Affymetrix"]*len(B),
                  index=combined.index, name="batch")
print("combined matrix (samples x genes):", combined.shape)
print(batch.value_counts().to_dict())

> **Exercise 4.1 — run PCA and produce the two-colouring plot.**
>
> Standardise the combined features (for visualisation only — flag it as *exploratory, not a
> modelling step*), run PCA to 2 components, and make **two scatter plots of the same points**:
> one coloured by `batch`, one coloured by the recurrence label (for METABRIC samples; GSE6532
> can be shown as "unlabelled"). Describe what dominates PC1.
>
> *Hints:* `from sklearn.preprocessing import StandardScaler`, `from sklearn.decomposition import PCA`.
> Drop genes with any NaN, then **select the top ~2,000 highly-variable genes** before PCA — running PCA on the whole transcriptome lets thousands of low-variance genes add noise rather than structure. *Hints:* `M = combined.dropna(axis=1)`, then `M = M[M.var().sort_values(ascending=False).head(2000).index]`.


In [ ]:
# TODO 4.1
# Standardise M = combined.dropna(axis=1) (exploratory only!), PCA -> 2 comps, print variance.
# Then TWO scatter plots of the SAME points: coloured by batch, and by recurrence label.
# (Align the recurrence label to the combined index; GSE6532 samples are "unlabelled".)
#
# from sklearn.preprocessing import StandardScaler
# from sklearn.decomposition import PCA
# ...


> **Exercise 4.2 — is batch confounded with outcome?** Quantify it: cross-tabulate `batch`
> against the recurrence label (METABRIC samples only, since GSE6532 is unlabelled here). State a
> verdict with evidence.
>
> **Exercise 4.3 (reasoning, write it).** If batch and outcome were *perfectly* confounded, explain
> why no correction method (e.g. ComBat) could rescue the analysis.


In [ ]:
# TODO 4.2 / 4.3
# Cross-tabulate batch vs recurrence label (METABRIC samples only) and write a verdict.
# Then answer 4.3 in the markdown cell below.
# tab = pd.crosstab(...); print(tab)


**My Section 4.3 answer (perfect confounding):**

*(TODO in the student notebook.)*


> **Discussion 4.** PCA shows structure but not its *cause*. What extra metadata would you request
> from the data generators to be sure PC1 is technical (platform) and not a genuine biological
> difference between the cohorts' patient populations?
>
> **Common misconception to retire:** that normalisation already "removed" batch, or that PCA is a
> correction step. PCA only *reveals*.


---
## Section 5 — Designing the splits  *(≈40 min — a core section)*

We now build splits **on the METABRIC labelled cohort** (the part with a usable label). A good split
is simultaneously **patient-level**, **stratified** by the label, and **batch-aware**.

> *Note on this data:* METABRIC is ~one-sample-per-patient, so true duplication isn't present —
> implement patient-level grouping anyway, as a discipline. The live constraint here is
> stratification of the minority class.


> **Exercise 5.1 — build a NAIVE random split first**, then measure how the minority class
> actually landed. Compare to your Section 3.2 prediction.
>
> *Hint:* `from sklearn.model_selection import train_test_split` with **no** `stratify`. Use a couple
> of different `random_state` values and watch the test-set positive rate wobble.


In [ ]:
# TODO 5.1
# Build a NAIVE 80/20 split (no stratify) for a few random_state values and print the test-set
# positive rate each time. Note how much it moves. Compare to your Section 3.2 prediction.
#
# from sklearn.model_selection import train_test_split
# Xlab = X_metabric.loc[y.index]
# for rs in [0,1,2,3]: ...


> **Exercise 5.2 — build the proper split and verify integrity.**
>
> Make a **patient-level, stratified** train/validation/test split (e.g. 60/20/20). Then run the
> integrity checks: (i) no patient appears in two splits; (ii) class proportions are preserved in
> each split; (iii) report the batch composition (here all METABRIC, but check the habit).
>
> *Hints:* `StratifiedGroupKFold` (groups = patient ID) or a two-step `train_test_split` with
> `stratify=y`. For METABRIC the patient ID is `PATIENT_ID` in `clin`.


In [ ]:
# TODO 5.2
# Build a patient-level, stratified 60/20/20 split.
# Verify: (i) zero patient overlap across splits; (ii) class proportions preserved;
#         (iii) batch composition per split.
#
# groups = clin.loc[y.index, "PATIENT_ID"]
# tr, tmp = train_test_split(..., stratify=y...) ; va, te = train_test_split(...)
# ... assert no overlap ; print proportions ; print batch per split


> **Exercise 5.3 — the leakage hunt (the payoff).**
>
> In writing: where in a typical pipeline would you normalise and feature-select, and **why must
> both happen *after* and *inside* the split**? Then point to the exact line in the naive workflow
> (below) where leakage occurs.


In [ ]:
# A DELIBERATELY BROKEN workflow -- find the leak (do NOT copy this into real work):
#
#   1  scaler = StandardScaler().fit(Xlab.values)         
#   2  Xall   = scaler.transform(Xlab.values)
#   3  top    = pick_top_genes_by_association(Xall, y)    
#   4  tr, te = train_test_split(Xall[:, top], y)        
#   5  model.fit(Xall[tr], y[tr]); model.score(Xall[te])   
print("Leak locations:  --- TO COMPLETE ---")

**My Section 5.3 answer:**

*(TODO in the student notebook — explain where normalisation & feature selection belong, and name
the leaking lines above.)*


> **Discussion 5.** Why is an independent **external cohort** worth more than any internal split?
> What would you look for in a candidate validation cohort — platform, patient population, processing,
> endpoint definition?


---
## Section 6 — Reflection: the data-readiness memo  *(≈15 min — main assessable artefact)*

Write a **200–300 word "data readiness" memo**. Is this dataset fit for building a *trustworthy*
recurrence biomarker? Cover its risks — sample size, class imbalance, batch confounding across the
two platforms, label/censoring ambiguity — and say what you would fix or request before any modelling.
This is your honest analyst's verdict.


**Data-readiness memo (200–300 words):**

*(TODO — write your verdict here.)*


---
### Deliverables checklist

- [ ] Written prediction-task statement (Section 0)
- [ ] Confirmed matrix orientation + clean joined table (Section 1)
- [ ] Clean binary label vector + label-decisions log incl. censoring policy (Section 2)
- [ ] Class-balance figure + interpretation (Section 3)
- [ ] Two-colouring PCA plots + batch/confounding verdict (Section 4)
- [ ] Patient-level, stratified, batch-aware split + integrity-check table (Section 5)
- [ ] Naive-vs-proper split comparison + identified leakage point (Section 5)
- [ ] Final data-readiness memo (Section 6)

> **Remember the message of the whole course:** *building a trustworthy predictive biomarker is more
> important than choosing a sophisticated algorithm.* Everything you did today is that rigour, before
> a single model is trained.


---
## Checkpoint — save the prepared cohort for Lessons 2–5

The later lessons don't re-derive the cohort; they **load what we save here**. This cell builds the
canonical HR+/HER2− cohort (binary 60-month recurrence label, top-2000-variance genes, a stratified
60/20/20 split) and writes it to the shared `datasets/derived/` folder as `lesson01_cohort`. Every
later lesson reuses this exact cohort **and this exact split**.

In [ ]:
# === Save the prepared cohort for the later lessons ==========================
# Lessons 2-5 LOAD this checkpoint (they do NOT re-download or re-derive it). We build the
# canonical HR+/HER2- cohort with the binary 60-month recurrence label, keep the top-2000
# highest-variance genes, and a stratified 60/20/20 split — then save all of it.
from sklearn.model_selection import train_test_split as _tts

def _build_cohort(expr, clin):
    X = expr.T.apply(pd.to_numeric, errors="coerce"); X.index.name = "SAMPLE_ID"
    common = sorted(set(X.index) & set(clin.index)); X, clin = X.loc[common], clin.loc[common]
    hrpos = clin.get("ER_STATUS").eq("Positive") | clin.get("PR_STATUS").eq("Positive")
    her2neg = clin.get("HER2_STATUS").eq("Negative")
    m = (hrpos & her2neg).fillna(False); X, clin = X.loc[m], clin.loc[m]
    status = next(c for c in ["RFS_STATUS", "DFS_STATUS"] if c in clin.columns)
    months = next(c for c in ["RFS_MONTHS", "DFS_MONTHS"] if c in clin.columns)
    rec = clin[status].astype(str).str.startswith("1"); mo = pd.to_numeric(clin[months], errors="coerce")
    y = pd.Series(index=clin.index, dtype="float")
    y[rec & (mo <= 60)] = 1; y[~rec & (mo >= 60)] = 0; y[rec & (mo > 60)] = 0
    keep = y.notna(); X, clin, y = X.loc[keep], clin.loc[keep], y[keep].astype(int)
    top = X.var().sort_values(ascending=False).head(2000).index
    return X[top], clin, y

Xc, clinc, yc = _build_cohort(metabric_expr, metabric_clin)
_idx = yc.index.to_numpy()
_tr, _tmp = _tts(_idx, test_size=0.40, random_state=0, stratify=yc.loc[_idx])
_va, _te  = _tts(_tmp, test_size=0.50, random_state=0, stratify=yc.loc[_tmp])
save_checkpoint("lesson01_cohort", X=Xc, clin=clinc, y=yc, tr=_tr, va=_va, te=_te)
print(f"cohort {Xc.shape[0]} patients x {Xc.shape[1]} genes | "
      f"split train {len(_tr)} val {len(_va)} test {len(_te)} | prevalence {yc.mean():.1%}")
